In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
import json

In [6]:
with open("/content/exercises.json", "r", encoding="utf-8") as f:
    data = json.load(f)

exercises = data["exercises"]

# Chuyển sang DataFrame để xem / phân tích
df = pd.DataFrame(exercises)
print(df.head())

# Lưu ra CSV để dùng nhanh
df.to_csv("exercises.csv", index=False, encoding="utf-8-sig")

     category                             description    equipment  \
0    strength      Sit-Up performed 3/4 of the way up       [none]   
1  stretching       Hamstring stretch with legs split       [none]   
2    strength         Crunch performed with a machine    [machine]   
3    strength  Rolling from a knee plank to extension      [other]   
4  stretching              One leg adductor foam roll  [foam roll]   

                                        instructions             name  \
0  [Lie down on the floor and secure your feet. Y...       3/4 Sit-Up   
1  [Lie on your back, with one leg extended strai...  90/90 Hamstring   
2  [Select a light resistance and sit down on the...   Machine Crunch   
3  [Hold the Ab Roller with both hands and kneel ...        Ab Roller   
4  [Lie face down with one leg on a foam roll., R...         Adductor   

  primary_muscles secondary_muscles variations_on  \
0           [abs]                []      [sit-up]   
1    [hamstrings]          [calves

In [7]:
# Select the relevant columns for the fitness exercise project
relevant_columns = ['name', 'category', 'primary_muscles', 'equipment', 'description', 'instructions', 'video']
df_processed = df[relevant_columns].copy()

# Display the first few rows of the processed DataFrame
print("DataFrame sau khi tiền xử lý:")
display(df_processed.head())

DataFrame sau khi tiền xử lý:


,name,category,primary_muscles,equipment,description,instructions,video
0,3/4 Sit-Up,strength,[abs],[none],Sit-Up performed 3/4 of the way up,[Lie down on the floor and secure your feet. Y...,https://www.youtube.com/watch?v=wm47Swzn_98
1,90/90 Hamstring,stretching,[hamstrings],[none],Hamstring stretch with legs split,"[Lie on your back, with one leg extended strai...",https://www.youtube.com/watch?v=h_yZV27H684
2,Machine Crunch,strength,[abs],[machine],Crunch performed with a machine,[Select a light resistance and sit down on the...,https://www.youtube.com/watch?v=zDSvdyyxH4w
3,Ab Roller,strength,[abs],[other],Rolling from a knee plank to extension,[Hold the Ab Roller with both hands and kneel ...,https://www.youtube.com/watch?v=aEgyCi1J1qI&t=82s
4,Adductor,stretching,[adductors],[foam roll],One leg adductor foam roll,"[Lie face down with one leg on a foam roll., R...",https://www.youtube.com/watch?v=h1xQabdHdes


In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# 1️⃣ Làm sạch dữ liệu văn bản
df_processed["description"] = df_processed["description"].fillna("").astype(str)
df_processed["instructions"] = df_processed["instructions"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (str(x) if pd.notna(x) else "")
)

# 2️⃣ Tạo văn bản tổng hợp để embed
texts = (df_processed["description"] + " " + df_processed["instructions"]).tolist()

# Xoá phần tử trống
texts = [t if isinstance(t, str) else "" for t in texts]

# 3️⃣ Tạo embedding
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, show_progress_bar=True)

# 4️⃣ Gom cụm thành 3 độ khó (0, 1, 2)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_processed["difficulty"] = kmeans.fit_predict(embeddings)

# 5️⃣ Xem kết quả
print(df_processed[["name", "difficulty"]].head())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

              name  difficulty
0       3/4 Sit-Up           1
1  90/90 Hamstring           1
2   Machine Crunch           1
3        Ab Roller           1
4         Adductor           1


In [9]:
display(df_processed.head(20))


,name,category,primary_muscles,equipment,description,instructions,video,difficulty
0,3/4 Sit-Up,strength,[abs],[none],Sit-Up performed 3/4 of the way up,Lie down on the floor and secure your feet. Yo...,https://www.youtube.com/watch?v=wm47Swzn_98,1
1,90/90 Hamstring,stretching,[hamstrings],[none],Hamstring stretch with legs split,"Lie on your back, with one leg extended straig...",https://www.youtube.com/watch?v=h_yZV27H684,1
2,Machine Crunch,strength,[abs],[machine],Crunch performed with a machine,Select a light resistance and sit down on the ...,https://www.youtube.com/watch?v=zDSvdyyxH4w,1
3,Ab Roller,strength,[abs],[other],Rolling from a knee plank to extension,Hold the Ab Roller with both hands and kneel o...,https://www.youtube.com/watch?v=aEgyCi1J1qI&t=82s,1
4,Adductor,stretching,[adductors],[foam roll],One leg adductor foam roll,Lie face down with one leg on a foam roll. Rot...,https://www.youtube.com/watch?v=h1xQabdHdes,1
5,Adductor/Groin,stretching,[adductors],[none],Groin Split Leg Stretch,Lie on your back with your feet raised towards...,https://www.youtube.com/watch?v=ScGo0VV85qA,1
6,Advanced Kettlebell Windmill,strength,[abs],[kettlebell],,Clean and press a kettlebell overhead with one...,NaN,2
7,Air Bike,strength,[abs],[none],,Lie flat on the floor with your lower back pre...,NaN,1
8,All Fours Quad Stretch,stretching,[quads],[none],,"Start off on your hands and knees, then lift y...",NaN,1
9,Alternate Hammer Curl,strength,"[biceps, brachialis]",[dumbbell],Hammer curl performed with dumbbells and one a...,Stand up with your torso upright and a dumbbel...,https://www.youtube.com/watch?v=zC3nLlEvin4,0


In [10]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd

class ExerciseFilter:
    def __init__(self, dataframe):
        # Bỏ cột không cần
        self.df = dataframe.drop(columns=['description'], errors='ignore')

        # MET trung bình cho từng nhóm category
        self.met_by_category = {
            "strength": 6.0,
            "stretching": 2.3,
            "plyometrics": 8.0,
            "strongman": 9.0,
            "cardio": 8.5,
            "olympic weightlifting": 6.8,
            "crossfit": 10.0,
            "calisthenics": 5.0
        }

        # Gợi ý rep và set cho từng nhóm category
        self.reps_sets_by_category = {
            "strength": {"reps": "8-12", "sets": "3-5"},
            "stretching": {"reps": "-", "sets": "2-3 (30–60s hold)"},
            "plyometrics": {"reps": "10-20", "sets": "3-4"},
            "strongman": {"reps": "4-8", "sets": "3-5"},
            "cardio": {"reps": "-", "sets": "20–60 phút"},
            "olympic weightlifting": {"reps": "1-5", "sets": "3-6"},
            "crossfit": {"reps": "tùy WOD", "sets": "3-5 rounds"},
            "calisthenics": {"reps": "10-20", "sets": "3-4"}
        }

    def filter_exercises(self, muscle, equipment=None, difficulty=None):
        """
        Lọc bài tập theo nhóm cơ, dụng cụ và độ khó.
        """
        filtered_df = self.df[self.df['primary_muscles'].apply(lambda x: muscle.lower() in [m.lower() for m in x])].copy()

        if equipment is not None:
            filtered_df = filtered_df[filtered_df['equipment'].apply(lambda x: equipment.lower() in [e.lower() for e in x])]

        if difficulty is not None:
            filtered_df = filtered_df[filtered_df['difficulty'] == difficulty]

        # Thêm rep/set và MET tương ứng
        filtered_df["reps"] = filtered_df["category"].apply(lambda c: self.reps_sets_by_category.get(c, {}).get("reps", "-"))
        filtered_df["sets"] = filtered_df["category"].apply(lambda c: self.reps_sets_by_category.get(c, {}).get("sets", "-"))
        filtered_df["MET"] = filtered_df["category"].apply(lambda c: self.met_by_category.get(c, 6.0))

        return filtered_df

    def estimate_calories_burned(self, exercise_name, duration_minutes, weight_kg):
        """
        Ước tính lượng calo tiêu thụ dựa trên MET của category.
        """
        row = self.df[self.df['name'].str.lower() == exercise_name.lower()]
        if row.empty:
            print(f"⚠️ Không tìm thấy bài tập '{exercise_name}'.")
            return 0

        category = row.iloc[0]['category'].lower()
        MET = self.met_by_category.get(category, 6.0)

        # Công thức chuẩn: Calo = MET * 3.5 * cân nặng (kg) / 200 * thời gian (phút)
        calories = MET * 3.5 * weight_kg / 200 * duration_minutes
        return calories


# ------------------ 🌟 Ví dụ dùng ------------------
# Giả sử bạn đã có df_processed có cột ['name', 'category', 'primary_muscles', 'equipment', 'difficulty']
# Nếu chưa có 'difficulty', có thể dùng KMeans để gán tạm 3 mức độ (0, 1, 2)

if 'difficulty' not in df_processed.columns:
    print("⚠️ Cảnh báo: Thiếu cột 'difficulty' — nên thêm bằng clustering trước.")
else:
    exercise_filter = ExerciseFilter(df_processed)

    print("\n🏋️‍♂️ Bài tập cơ bụng, không dụng cụ, độ khó 1:")
    abs_ex = exercise_filter.filter_exercises(muscle='abs', equipment='none', difficulty=1)
    display(abs_ex[['name', 'category', 'reps', 'sets', 'MET']])

    print("-" * 50)

    print("\n💪 Tất cả bài tập cơ ngực:")
    chest_ex = exercise_filter.filter_exercises(muscle='chest')
    display(chest_ex[['name', 'category', 'reps', 'sets', 'MET']])

    print("-" * 50)

    print("\n🦾 Bài tập tay trước với tạ đơn:")
    biceps_ex = exercise_filter.filter_exercises(muscle='biceps', equipment='dumbbell')
    display(biceps_ex[['name', 'category', 'reps', 'sets', 'MET']])

    print("-" * 50)

    # Ước tính calo
    cal = exercise_filter.estimate_calories_burned('Plank', duration_minutes=5, weight_kg=70)
    print(f"🔥 Ước tính calo tiêu thụ khi plank 5 phút (70kg): {cal:.2f} kcal")



🏋️‍♂️ Bài tập cơ bụng, không dụng cụ, độ khó 1:


,name,category,reps,sets,MET
0,3/4 Sit-Up,strength,8-12,3-5,6.0
7,Air Bike,strength,8-12,3-5,6.0
10,Alternate Heel Touchers,strength,8-12,3-5,6.0
78,Bent-Knee Hip Raise,strength,8-12,3-5,6.0
98,Bottoms Up,strength,8-12,3-5,6.0
106,Butt-Ups,strength,8-12,3-5,6.0
178,Cocoons,strength,8-12,3-5,6.0
181,Cross-Body Crunch,strength,8-12,3-5,6.0
186,Crunch - Hands Overhead,strength,8-12,3-5,6.0
187,Crunch - Legs On Exercise Ball,strength,8-12,3-5,6.0


--------------------------------------------------

💪 Tất cả bài tập cơ ngực:


,name,category,reps,sets,MET
15,Alternating Floor Press,strength,8-12,3-5,6.0
26,Around The Worlds,strength,8-12,3-5,6.0
44,Barbell Bench Press,strength,8-12,3-5,6.0
50,Barbell Guillotine Bench Press,strength,8-12,3-5,6.0
53,Barbell Incline Bench Press,strength,8-12,3-5,6.0
...,...,...,...,...,...
805,Suspended Push-Up,strength,8-12,3-5,6.0
809,Svend Press,strength,8-12,3-5,6.0
853,Wide-Grip Barbell Bench Press,strength,8-12,3-5,6.0
854,Wide-Grip Decline Barbell Bench Press,strength,8-12,3-5,6.0


--------------------------------------------------

🦾 Bài tập tay trước với tạ đơn:


,name,category,reps,sets,MET
9,Alternate Hammer Curl,strength,8-12,3-5,6.0
11,Alternate Incline Dumbbell Curl,strength,8-12,3-5,6.0
180,Concentration Curls,strength,8-12,3-5,6.0
182,Cross Body Hammer Curl,strength,8-12,3-5,6.0
220,Dumbbell Alternate Bicep Curl,strength,8-12,3-5,6.0
223,Dumbbell Bicep Curl,strength,8-12,3-5,6.0
237,Dumbbell Prone Incline Curl,strength,8-12,3-5,6.0
273,Flexor Incline Dumbbell Curls,strength,8-12,3-5,6.0
308,Hammer Curl,strength,8-12,3-5,6.0
340,Incline Dumbbell Curl,strength,8-12,3-5,6.0


--------------------------------------------------
🔥 Ước tính calo tiêu thụ khi plank 5 phút (70kg): 36.75 kcal
